# UME quick start

This notebook loads the released base UME weights, extracts embeddings from EDA windows, and fine-tunes a classifier with `UMEPipeline`.

It uses random data so it runs anywhere. Replace `X` and `y` with your own windows: **60 s at 4 Hz (240 samples) with 3 channels (mixed, phasic, tonic)**.

In [1]:
import numpy as np
import torch

from ume import UMEPipeline

torch.manual_seed(0)
np.random.seed(0)

## 1. Load the pretrained model

`artifacts/ume_base_inference.pt` holds only the weights and the model config, so it loads with `torch.load(weights_only=True)`. `artifacts/ume_base_training.ckpt` is the full PyTorch Lightning training checkpoint and loads the same model.

In [2]:
pipeline = UMEPipeline.from_pretrained(
    model_name="efficientnet",
    weights_path="../artifacts/ume_base_inference.pt",
    device_map="auto",  # MPS > CUDA > CPU
)
print(pipeline.model_config)

{'in_channels': 3, 'stem_channels': 64, 'kernel_size': 3, 'head_channels': 248, 'embedding_dim': 64, 'num_mbconv_blocks': 16, 'mbconv_channels': 64, 'seq_len': 240, 'dropout': 0.1, 'stride': 2}


## 2. Extract embeddings

In [3]:
x = torch.randn(8, 240, 3)  # (batch, seq_len, channels)
embeddings = pipeline.embed(x)
print(embeddings.shape)  # (8, embedding_dim)

torch.Size([8, 64])


## 3. Fine-tune a classifier

`fit` attaches a `LayerNorm -> Linear` head to the encoder. `fine_tuning_type="head"` trains only the head (a linear probe); `"full"` also updates the encoder.

In [4]:
X = torch.randn(128, 240, 3)
y = np.random.randint(0, 2, size=128)

pipeline.fit(
    X, y,
    fine_tuning_type="head",
    num_epochs=5,
    validation_split=0.2,
    verbose=True,
)

proba = pipeline.predict_proba(X[:4])  # column j is pipeline.classes_[j]
preds = pipeline.predict(X[:4])
print(proba.shape, preds)

Holding out 26 samples (20%) for validation.


Epoch 1/5 - loss: 0.8357 - macro_f1: 0.3544 - val_macro_f1: 0.3500
Epoch 2/5 - loss: 0.7905 - macro_f1: 0.3544 - val_macro_f1: 0.3500


Epoch 3/5 - loss: 0.7515 - macro_f1: 0.3544 - val_macro_f1: 0.3500
Epoch 4/5 - loss: 0.7309 - macro_f1: 0.3544 - val_macro_f1: 0.3500


Epoch 5/5 - loss: 0.7146 - macro_f1: 0.3544 - val_macro_f1: 0.3500


(4, 2) [1 1 1 1]
